In [1]:
import json
import pandas as pd

# 1. 파일 경로 (본인의 파일 경로로 수정)
file_path = '../SSU_Datathon2025_공학분야_62199.json'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if "NODE_LIST" in data:
        df = pd.DataFrame(data["NODE_LIST"])
        
        # 필수 컬럼 확인
        required_cols = ['PBSH', 'NODE_CLSS_02', 'IPRD_NM']
        if all(col in df.columns for col in required_cols):
            
            # 2. 전처리: 발행년월(PBSH)에서 년도(Year) 4자리 추출
            #    (문자열로 변환 후 앞 4글자 슬라이싱)
            df['Year'] = df['PBSH'].astype(str).str.strip().str[:4]
            
            # 3. 그룹화: [년도, 중분류] 별로 IPRD_NM(발행기관)의 고유값(unique) 추출
            #    as_index=False를 하면 데이터프레임 형태로 깔끔하게 나옵니다.
            result = df.groupby(['Year', 'NODE_CLSS_02'])['IPRD_NM'].unique()
            
            # 4. 결과 출력
            print("--- [년도 x 중분류 별 학회(발행기관) 목록] ---")
            
            for (year, clss), societies in result.items():
                # societies는 numpy array 형태이므로 리스트로 변환
                society_list = list(societies)
                count = len(society_list)
                
                print(f"\n📌 {year}년 | {clss} (총 {count}개 기관)")
                print(f"   ㄴ {', '.join(society_list)}")
                
        else:
            missing = [col for col in required_cols if col not in df.columns]
            print(f"오류: 필수 컬럼이 누락되었습니다. {missing}")

    else:
        print("오류: NODE_LIST 키가 없습니다.")

except Exception as e:
    print(f"에러 발생: {e}")

--- [년도 x 중분류 별 학회(발행기관) 목록] ---

📌 2021년 | 건축공학 (총 21개 기관)
   ㄴ 대한토목학회, 한국건축친환경설비학회, 대한국토·도시계획학회, 한국지반환경공학회, 대한공간정보학회, 한국콘크리트학회, 한국생태환경건축학회, 한국암반공학회, 한국지반공학회, 대한환경공학회, 대한건축학회, 서울대학교 환경대학원, 새건축사협의회, 한국건설순환자원학회, 한국강구조학회, 한국해안해양공학회, 한국지반신소재학회, 한국자원공학회, 한국가스학회, 한국측량학회, 한국지열·수열에너지학회

📌 2021년 | 공학 일반 (총 3개 기관)
   ㄴ 한국산학기술학회, 한국산업정보학회, 한국센서학회

📌 2021년 | 기계공학 (총 28개 기관)
   ㄴ 대한설비공학회, 대한기계학회, 한국추진공학회, 한국비파괴검사학회, 한국자동차공학회, 한국에너지기후변화학회, 한국생산제조학회, 한국항공우주학회, 한국표면공학회, Korean Society for Precision Engineering, 한국트라이볼로지학회, 한국기계가공학회, 한국유체기계학회, 한국수소및신에너지학회, 대한용접·접합학회, 한국연소학회, 한국동력기계공학회, 한국전산유체공학회, 한국소음진동공학회, 한국환경에너지공학회, (사)한국CDE학회, 유공압건설기계학회, 한국기계연구원, 한국항공우주연구원, 항공우주시스템공학회, 한국마린엔지니어링학회, 한국자동차안전학회, 한국항공협회

📌 2021년 | 기타 공학 (총 5개 기관)
   ㄴ 한국철도학회, 대한교통학회, 한국위험물학회, 한국게임학회, 한국재활복지공학회

📌 2021년 | 산업공학 (총 5개 기관)
   ㄴ 한국신뢰성학회, 한국지능정보시스템학회, 대한산업공학회, 한국SCM학회, 대한인간공학회

📌 2021년 | 재료·에너지공학 (총 7개 기관)
   ㄴ 한국태양광발전학회, 한국태양에너지학회, 한국압력기기공학회, 한국에너지학회, 한국염색가공학회, 한국신재생에너지학회, 한국소성·가공학회

📌 2021년 | 전기전자공학 (총 23개 기관)
   ㄴ 제어로봇시스

In [8]:
import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}
# -----------------------------------------------------------

def normalize_name(name):
    """매칭 확률을 높이기 위해 공백과 특수문자를 제거하는 함수"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def main():
    # =======================================================
    # STEP 1: JSON 파일 로드 및 '발행기관(IPRD_NM)' 집계
    # =======================================================
    print(f"📂 JSON 파일 로드 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        if "NODE_LIST" not in data:
            print("❌ 오류: JSON 파일에 'NODE_LIST' 키가 없습니다.")
            return

        df_origin = pd.DataFrame(data["NODE_LIST"])
        df_origin['Year'] = df_origin['PBSH'].astype(str).str.strip().str[:4]
        
        # 발행기관(IPRD_NM) 기준 집계
        df_grouped = df_origin.groupby(['Year', 'NODE_CLSS_02']).agg(
            논문수=('NODE_ID', 'count'),
            학회_리스트=('IPRD_NM', lambda x: list(x.unique())) 
        ).reset_index()
        
        print(f"✅ JSON 데이터 집계 완료 (총 {len(df_grouped)}개 그룹)")

    except Exception as e:
        print(f"❌ JSON 처리 중 오류 발생: {e}")
        return

    # =======================================================
    # STEP 2: 인용지수 엑셀 로드
    # =======================================================
    print("📂 인용지수 엑셀 파일 로드 및 매핑 테이블 생성 중...")
    if_data = {}

    def load_xls_file(year, path):
        if not os.path.exists(path): return {}
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # IF 컬럼 찾기
            if_cols = [c for c in df.columns if '2년' in c and 'IF' in c]
            if not if_cols: return {}
            if_col = if_cols[0]
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            name_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

            # 점수 숫자 변환
            df[if_col] = pd.to_numeric(df[if_col], errors='coerce').fillna(0)
            
            # 매핑 테이블 생성
            mapping = {}
            for idx, row in df.iterrows():
                org_name = row[name_col]
                score = row[if_col]
                norm_key = normalize_name(org_name)
                
                if norm_key:
                    # 이미 있으면 더 높은 점수 유지
                    if norm_key not in mapping:
                        mapping[norm_key] = score
                    else:
                        mapping[norm_key] = max(mapping[norm_key], score)
            return mapping
            
        except Exception as e:
            print(f"   ⚠️ {year}년 파일 로드 실패: {e}")
            return {}

    for y in [2021, 2022, 2023, 2024]:
        if_data[y] = load_xls_file(y, file_paths_if.get(y, ''))
    if_data[2025] = if_data.get(2024, {})

    # =======================================================
    # STEP 3: 순위 매기기 (소수점 2자리 포맷팅 적용)
    # =======================================================
    print("📊 발행기관 기준 순위 산출 중...")

    def rank_journals(row):
        try:
            year = int(row['Year'])
        except:
            return ""
        
        org_list = row['학회_리스트']
        if not org_list: return ""
        
        year_map = if_data.get(year, {})
        
        ranked_list = []
        for org_name in org_list:
            clean_org_name = str(org_name).strip()
            norm_name = normalize_name(clean_org_name)
            score = year_map.get(norm_name, 0)
            
            ranked_list.append((clean_org_name, score))
        
        # 점수 내림차순 정렬
        ranked_list.sort(key=lambda x: (-x[1], x[0]))
        
        # [수정된 부분] score:.2f -> 소수점 2자리까지만 출력
        return "\n".join([f"{i+1}. {name} ({score:.2f})" for i, (name, score) in enumerate(ranked_list)])

    df_grouped['학회_순위(IF)'] = df_grouped.apply(rank_journals, axis=1)

    # =======================================================
    # STEP 4: 저장
    # =======================================================
    output_file = 'result_ranked_by_org.csv'
    final_columns = ['Year', 'NODE_CLSS_02', '논문수', '학회_순위(IF)']
    
    try:
        df_grouped[final_columns].to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n🎉 완료! '{output_file}'에 저장되었습니다.")
        print(df_grouped[final_columns].head())
        
    except Exception as e:
        print(f"❌ 저장 실패: {e}")

if __name__ == "__main__":
    main()

📂 JSON 파일 로드 중... (../SSU_Datathon2025_공학분야_62199.json)
✅ JSON 데이터 집계 완료 (총 50개 그룹)
📂 인용지수 엑셀 파일 로드 및 매핑 테이블 생성 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📊 발행기관 기준 순위 산출 중...

🎉 완료! 'result_ranked_by_org.csv'에 저장되었습니다.
   Year NODE_CLSS_02   논문수                                          학회_순위(IF)
0  2021         건축공학  2402  1. 대한국토·도시계획학회 (1.35)\n2. 대한공간정보학회 (1.07)\n3. ...
1  2021        공학 일반  1128  1. 한국산학기술학회 (1.00)\n2. 한국산업정보학회 (0.93)\n3. 한국센...
2  2021         기계공학  2635  1. 대한용접·접합학회 (0.58)\n2. 한국수소및신에너지학회 (0.56)\n3....
3  2021        기타 공학   447  1. 대한교통학회 (1.04)\n2. 한국철도학회 (0.41)\n3. 한국재활복지공...
4  2021         산업공학   289  1. 한국지능정보시스템학회 (1.15)\n2. 한국SCM학회 (0.78)\n3. 한...


In [7]:
# 매칭되지 않은 학회를 연도x중분류 별로 정리

import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}
# -----------------------------------------------------------

def normalize_name(name):
    """매칭 정확도를 높이기 위해 공백과 특수문자를 제거"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_excel_publishers(year, path):
    """엑셀 파일에서 발행기관 목록 추출"""
    if not os.path.exists(path):
        return set()
    try:
        df = pd.read_excel(path, engine='xlrd')
        df.columns = df.columns.str.replace('\n', '').str.strip()
        
        # 발행기관 관련 컬럼 찾기
        org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
        target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

        publishers = set()
        for name in df[target_col].dropna():
            publishers.add(normalize_name(name))
        return publishers
    except:
        return set()

def format_list_with_newlines(items):
    """
    리스트를 받아서 
    1. 항목A
    2. 항목B 
    형태로 줄바꿈(\n)하여 반환
    """
    if not items:
        return "-"
    # 번호 매기기 + 줄바꿈 문자(\n)로 연결
    return "\n".join([f"{i+1}. {item}" for i, item in enumerate(items)])

def main():
    # -------------------------------------------------------
    # STEP 1: 비교할 기준 데이터(Excel) 로드
    # -------------------------------------------------------
    print("📂 비교 기준 데이터(인용지수 엑셀) 로드 중...")
    if_db = {}
    for year in [2021, 2022, 2023, 2024]:
        if_db[year] = load_excel_publishers(year, file_paths_if.get(year, ''))
    if_db[2025] = if_db.get(2024, set())

    # -------------------------------------------------------
    # STEP 2: JSON 분석 및 결과 생성
    # -------------------------------------------------------
    print(f"📂 JSON 파일 분석 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if "NODE_LIST" in data:
            df = pd.DataFrame(data["NODE_LIST"])
            df['Year'] = df['PBSH'].astype(str).str.strip().str[:4]
            
            result = df.groupby(['Year', 'NODE_CLSS_02'])['IPRD_NM'].unique()
            output_rows = []
            
            for (year_str, clss), societies in result.items():
                try:
                    year_int = int(year_str)
                except:
                    year_int = 0
                
                valid_publishers = if_db.get(year_int, set())
                
                matched = []
                unmatched = []
                
                for society in societies:
                    norm_name = normalize_name(society)
                    if norm_name in valid_publishers:
                        matched.append(society)
                    else:
                        unmatched.append(society)
                
                total_cnt = len(societies)
                match_cnt = len(matched)
                match_rate = (match_cnt / total_cnt * 100) if total_cnt > 0 else 0
                
                # [수정됨] 여기서 줄바꿈 함수 적용
                row = {
                    'Year': year_str,
                    'Category': clss,
                    'Total_Count': total_cnt,
                    'Match_Rate(%)': round(match_rate, 1),
                    'Listed_Count': match_cnt,
                    'Listed_Societies(O)': format_list_with_newlines(matched),   # 줄바꿈 적용
                    'Unlisted_Count': len(unmatched),
                    'Unlisted_Societies(X)': format_list_with_newlines(unmatched) # 줄바꿈 적용
                }
                output_rows.append(row)
            
            # -------------------------------------------------------
            # STEP 3: CSV 파일 저장
            # -------------------------------------------------------
            if output_rows:
                result_df = pd.DataFrame(output_rows)
                
                cols = ['Year', 'Category', 'Total_Count', 'Match_Rate(%)', 
                        'Listed_Count', 'Unlisted_Count', 
                        'Listed_Societies(O)', 'Unlisted_Societies(X)']
                result_df = result_df[cols]
                
                output_filename = 'society_verification_formatted.csv'
                result_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
                
                print(f"\n🎉 저장 완료! '{output_filename}' 파일을 확인하세요.")
                print("💡 엑셀에서 파일을 연 뒤, 셀 서식에서 '텍스트 줄바꿈'을 켜주세요.")
            else:
                print("저장할 데이터가 없습니다.")

        else:
            print("오류: NODE_LIST 키가 없습니다.")

    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 비교 기준 데이터(인용지수 엑셀) 로드 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 JSON 파일 분석 중... (../SSU_Datathon2025_공학분야_62199.json)

🎉 저장 완료! 'society_verification_formatted.csv' 파일을 확인하세요.
💡 엑셀에서 파일을 연 뒤, 셀 서식에서 '텍스트 줄바꿈'을 켜주세요.


In [9]:
# 전체 데이터를 봤을 때 매칭되지 않은 학회를 리스트로 정리

import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}
# -----------------------------------------------------------

def normalize_name(name):
    """매칭 정확도를 높이기 위해 공백과 특수문자를 제거"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_excel_publishers(year, path):
    """엑셀 파일에서 발행기관 목록(정규화됨) 추출"""
    if not os.path.exists(path):
        return set()
    try:
        df = pd.read_excel(path, engine='xlrd')
        df.columns = df.columns.str.replace('\n', '').str.strip()
        
        # 발행기관 관련 컬럼 찾기
        org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
        target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

        publishers = set()
        for name in df[target_col].dropna():
            publishers.add(normalize_name(name))
        return publishers
    except:
        return set()

def main():
    print("🔍 미매칭 학회 추출 작업을 시작합니다...")

    # -------------------------------------------------------
    # 1. 인용지수 DB 로드 (Lookup Table)
    # -------------------------------------------------------
    if_db = {}
    for year in [2021, 2022, 2023, 2024]:
        if_db[year] = load_excel_publishers(year, file_paths_if.get(year, ''))
    
    # 2025년은 2024년 데이터로 대체
    if_db[2025] = if_db.get(2024, set())

    # -------------------------------------------------------
    # 2. JSON 파일 스캔 및 미매칭 학회 수집
    # -------------------------------------------------------
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if "NODE_LIST" in data:
            df = pd.DataFrame(data["NODE_LIST"])
            df['Year'] = df['PBSH'].astype(str).str.strip().str[:4]
            
            # (년도, 학회명) 쌍 중복 제거 후 순회
            # 이유: 같은 년도에 같은 학회 논문이 100개여도 한 번만 검사하면 됨
            unique_pairs = df[['Year', 'IPRD_NM']].drop_duplicates()
            
            unmatched_set = set() # 중복 제거를 위해 집합(Set) 사용
            
            for idx, row in unique_pairs.iterrows():
                try:
                    year_int = int(row['Year'])
                except:
                    year_int = 0
                
                org_name = row['IPRD_NM']
                norm_name = normalize_name(org_name)
                
                valid_publishers = if_db.get(year_int, set())
                
                # 매칭 실패 시 수집
                if norm_name not in valid_publishers:
                    unmatched_set.add(org_name)
            
            # -------------------------------------------------------
            # 3. 결과 저장
            # -------------------------------------------------------
            if unmatched_set:
                # 리스트로 변환 후 정렬 (가나다순)
                sorted_list = sorted(list(unmatched_set))
                
                output_df = pd.DataFrame(sorted_list, columns=['Unmatched_Society_Name'])
                
                output_file = 'unmatched_societies_list.csv'
                output_df.to_csv(output_file, index=False, encoding='utf-8-sig')
                
                print("\n" + "="*50)
                print(f"🎉 추출 완료! 총 {len(sorted_list)}개의 미매칭 학회가 발견되었습니다.")
                print(f"📂 저장된 파일: {output_file}")
                print("="*50)
                
                # 미리보기 (최대 10개)
                print("\n[미매칭 학회 예시]")
                for name in sorted_list[:10]:
                    print(f"- {name}")
                if len(sorted_list) > 10:
                    print("... (생략)")
            else:
                print("🎉 축하합니다! 모든 학회가 정상적으로 매칭되었습니다.")

        else:
            print("오류: NODE_LIST 키가 없습니다.")

    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

🔍 미매칭 학회 추출 작업을 시작합니다...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512

🎉 추출 완료! 총 24개의 미매칭 학회가 발견되었습니다.
📂 저장된 파일: unmatched_societies_list.csv

[미매칭 학회 예시]
- (사)한국CDE학회
- ICT플랫폼학회
- Korean Institute of Information Scientists and Engineers
- Korean Society for Precision Engineering
- 국방로봇학회
- 새건축사협의회
- 서울대학교 환경대학원
- 세종대학교 우주항공산업연구소
- 유공압건설기계학회
- 한국게임학회
... (생략)


In [11]:
import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = '../SSU_Datathon2025_공학분야_62199.json'

file_paths_if = {
    2021: '2021_인용지수_2년분.xls',
    2022: '2022_인용지수_2년분.xls',
    2023: '2023_인용지수_2년분.xls',
    2024: '2024_인용지수_2년분.xls'
}

# -----------------------------------------------------------
# 2. 사용자 수기 확인 내역 (매핑 테이블)
# -----------------------------------------------------------
manual_mapping = {
    "(사)한국CDE학회": "한국CDE학회",
    "ICT플랫폼학회": "아이씨티플랫폼학회",
    "유공압건설기계학회": "사단법인 유공압건설기계학회",
    "한국로봇학회(논문지)": "한국로봇학회",
    "한국염색가공학회": "한국염색가공학회",
    "한국위험물학회": "한국위험물학회",
    "한국자동차안전학회": "사단법인 한국자동차안전학회",
    "한국전자파학회JEES": "한국전자파학회",
    "한국정보통신학회JICCE": "한국정보통신학회",
    "한국컴퓨터그래픽스학회": "(사)한국컴퓨터그래픽스학회",
    "한국콘텐츠학회(IJOC)": "한국콘텐츠학회",
    "한국환경에너지공학회": "(사)한국환경에너지공학회"
}

# -----------------------------------------------------------
# 3. 유틸리티 함수
# -----------------------------------------------------------
def normalize_name(name):
    """매칭 확률을 높이기 위해 공백과 특수문자를 제거"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_if_database(file_paths):
    """모든 연도의 엑셀을 읽어서 IF DB 구축"""
    db = {}
    print("📂 인용지수 데이터베이스(Excel) 로드 중...")
    
    for year, path in file_paths.items():
        if not os.path.exists(path):
            db[year] = {}
            continue
            
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # IF 컬럼 찾기
            if_cols = [c for c in df.columns if '2년' in c and 'IF' in c]
            if not if_cols:
                db[year] = {}
                continue
            if_col = if_cols[0]
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

            # 점수 매핑 (Key: 정규화된 이름 -> Value: 점수)
            mapping = {}
            df[if_col] = pd.to_numeric(df[if_col], errors='coerce').fillna(0)
            
            for idx, row in df.iterrows():
                org_name = row[target_col]
                score = row[if_col]
                norm_name = normalize_name(org_name)
                
                if norm_name:
                    mapping[norm_name] = max(mapping.get(norm_name, 0), score)
            
            db[year] = mapping
            
        except Exception as e:
            print(f"   ❌ {year}년 로드 실패: {e}")
            db[year] = {}
            
    db[2025] = db.get(2024, {})
    return db

def main():
    # 1. 인용지수 DB 로드
    if_db = load_if_database(file_paths_if)
    
    # 2. JSON 로드
    print(f"📂 JSON 파일 분석 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if "NODE_LIST" not in data:
            print("오류: NODE_LIST 키가 없습니다.")
            return

        df = pd.DataFrame(data["NODE_LIST"])
        df['Year'] = df['PBSH'].astype(str).str.strip().str[:4]
        
        # 3. 그룹화: [년도, 중분류] 별 학회 목록 추출
        result_groups = df.groupby(['Year', 'NODE_CLSS_02'])['IPRD_NM'].unique()
        
        output_rows = []
        
        print("📊 순위 산출 및 CSV 생성 중...")
        
        for (year_str, clss), societies in result_groups.items():
            try:
                year_int = int(year_str)
            except:
                year_int = 0
            
            year_db = if_db.get(year_int, {})
            
            # 해당 그룹의 학회들에 대해 점수 조회
            ranked_list = []
            
            for original_name in societies:
                # [수정된 부분] 수기 매핑 적용
                if original_name in manual_mapping:
                    search_name = manual_mapping[original_name]
                else:
                    search_name = original_name
                
                # 정규화 후 DB 조회
                norm_name = normalize_name(search_name)
                score = year_db.get(norm_name, 0)
                
                # 리스트에 저장 (원본 이름, 점수)
                # 원본 이름을 출력해야 나중에 헷갈리지 않음
                ranked_list.append((original_name, score))
            
            # 정렬: 점수 내림차순 -> 이름 오름차순
            ranked_list.sort(key=lambda x: (-x[1], x[0]))
            
            # 출력용 문자열 생성 (줄바꿈 포함)
            # 예: "1. 대한전기학회 (0.85)\n2. 한국통신학회 (0.52)"
            rank_str_parts = []
            for i, (name, score) in enumerate(ranked_list):
                rank_str_parts.append(f"{i+1}. {name} ({score:.2f})")
            
            final_rank_str = "\n".join(rank_str_parts)
            
            # 행 데이터 추가
            output_rows.append({
                'Year': year_str,
                'Category': clss,
                'Total_Societies': len(societies),
                'Society_Rankings(IF)': final_rank_str
            })
            
        # 4. CSV 저장
        if output_rows:
            result_df = pd.DataFrame(output_rows)
            output_file = 'society_rankings_corrected.csv'
            
            result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            
            print("\n" + "="*60)
            print(f"🎉 순위표 생성 완료!")
            print(f"📂 저장된 파일: {output_file}")
            print("💡 엑셀에서 파일을 열고 '텍스트 줄바꿈'을 켜주세요.")
            print("="*60)
            
            # 미리보기
            print(result_df.head())
            
    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 인용지수 데이터베이스(Excel) 로드 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 JSON 파일 분석 중... (../SSU_Datathon2025_공학분야_62199.json)
📊 순위 산출 및 CSV 생성 중...

🎉 순위표 생성 완료!
📂 저장된 파일: society_rankings_corrected.csv
💡 엑셀에서 파일을 열고 '텍스트 줄바꿈'을 켜주세요.
   Year Category  Total_Societies  \
0  2021     건축공학               21   
1  2021    공학 일반                3   
2  2021     기계공학               28   
3  2021    기타 공학                5   
4  2021     산업공학                5   

                                Society_Rankings(IF)  
0  1. 대한국토·도시계획학회 (1.35)\n2. 대한공간정보학회 (1.07)\n3. ...  
1  1. 한국산학기술학회 (1.00)\n2. 한국산업정보학회 (0.93)\n3. 한국센...  
2  1. (사)한국CDE학회 (1.13)\n2. 대한용접·접합학회 (0.58)\n3. ...  
3  1. 대한교통학회 (1.04)\n2. 한국철도학회 (0.41)\n3. 한국재활복지공...  
4  1. 한국지능정보시스템학회 (1.15)\n2. 